# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "documents/ai_report_2025.pdf"   # "documents/managing_oneself.pdf"
docs = PyPDFLoader(pdf_path).load() # returns a list of Document objects, each Document has: page_content (string text), metadata (a dict)

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Pages: {len(docs)}")
print(document_text[:800])

# content now in document_text variable
# read svery pages, eval with same pages.  


Pages: 26
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positio


In [3]:
print(type(docs))           # list
print(type(docs[0]))        # Document
print(docs[1].metadata)     # dict


<class 'list'>
<class 'langchain_core.documents.base.Document'>
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-07-13T21:18:19-07:00', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_method': 'Privileged', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_enabled': 'True', 'author': 'Aditya Challapally', 'moddate': '2025-07-13T21:18:19-07:00', 'source': 'documents/ai_report_2025.pdf', 'total_pages': 26, 'page': 1, 'page_label': '2'}


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [13]:
from openai import OpenAI
from pydantic import BaseModel, Field
import os

gateway_key = os.getenv("API_GATEWAY_KEY", "").strip()
print("gateway key loaded:", bool(gateway_key), "len:", len(gateway_key))

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="placeholder",
    default_headers={"x-api-key": gateway_key},
)

class SummaryCore(BaseModel):  # schema for what the model should generate
    Author: str = Field(description="Author of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(description="Why this article is relevant for an AI professional (max one paragraph)")
    Summary: str = Field(description="Concise summary, max 1000 tokens")
    Tone: str = Field(description="Writing tone used in the summary")

class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

tone = "Formal Academic Writing"
developer_instructions = f"""
You are an expert summarizer.
Return structured output only.
Constraints:
- Relevance must be one paragraph or shorter.
- Summary must be concise and <= 1000 tokens.
- Use this tone exactly: {tone}.
- Do not add facts not present in the provided article text.
"""

# Alternative approach: limit context size to reduce tokens/cost if needed.
# MAX_CHARS = 10_000
# context = document_text[:MAX_CHARS]
# For this assignment we instead use the full document_text to ensure complete summaries.

context = document_text

user_prompt = f"""
Extract author/title and summarize this article.

ARTICLE TEXT:
{context}
"""

response = client.responses.parse(
    model="gpt-4o-mini",  # not GPT-5 family
    input=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_prompt},
    ],
    text_format=SummaryCore,
)

core = response.output_parsed

summary_output = SummaryOutput(
    Author=core.Author,
    Title=core.Title,
    Relevance=core.Relevance,
    Summary=core.Summary,
    Tone=core.Tone,
    InputTokens=response.usage.input_tokens,
    OutputTokens=response.usage.output_tokens,
)


print("Summary Output:")
print(summary_output.model_dump_json(indent=4))

gateway key loaded: True len: 20
Summary Output:
{
    "Author": "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
    "Title": "The GenAI Divide: State of AI in Business 2025",
    "Relevance": "This article is crucial for AI professionals as it identifies key barriers and potential strategies in the AI adoption landscape, highlighting the distinction between successful and unsuccessful implementations in enterprises. It provides insights on the differing impacts of generative AI tools across industries and informs future AI integration strategies for effective business transformation.",
    "Summary": "The report outlines a significant disparity, termed the 'GenAI Divide,' where 95% of organizations see no return on their AI investments despite substantial funding of $30–40 billion in generative AI (GenAI) technologies. High adoption rates of general-purpose tools, such as ChatGPT, contrast starkly with low transformation results as only 5% of enterprise-grade pilots

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [18]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
from pydantic import BaseModel
import os
import time

class EvaluationOutput(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


def build_eval_model(gateway_key: str):
    gateway_key = (gateway_key or "").strip()
    if gateway_key:
        return GPTModel(
            model="gpt-4o-mini",
            _openai_api_key="placeholder",  # required by deepeval client init
            base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
            default_headers={"x-api-key": gateway_key},
            temperature=0,
        )
    return GPTModel(model="gpt-4o-mini", temperature=0)


def build_eval_source_from_pages(
    docs,
    max_chars=48000,
    head_pages=2,
    tail_pages=2,
    middle_pages=8,
    min_page_chars=200,
):
    pages = [d.page_content.strip() for d in docs if len(d.page_content.strip()) >= min_page_chars]
    if not pages:
        return ""

    full_text = "\n\n".join(pages)
    if len(full_text) <= max_chars:
        return full_text

    n = len(pages)

    # Always include first and last pages
    idx = set(range(min(head_pages, n)))
    idx.update(range(max(0, n - tail_pages), n))

    # Evenly sample middle pages
    left = min(head_pages, n)
    right = max(0, n - tail_pages)
    middle_pool = list(range(left, right))
    k = min(middle_pages, len(middle_pool))

    if k == 1:
        idx.add(middle_pool[len(middle_pool) // 2])
    elif k > 1:
        for i in range(k):
            pos = round(i * (len(middle_pool) - 1) / (k - 1))
            idx.add(middle_pool[pos])

    selected = []
    total = 0
    for i in sorted(idx):
        chunk = f"[Page {i + 1}]\n{pages[i]}"
        if total + len(chunk) + 2 > max_chars:
            break
        selected.append(chunk)
        total += len(chunk) + 2

    return "\n\n".join(selected)




def measure_with_backoff(metric, test_case, metric_name: str, max_retries: int = 5, base_sleep: int = 6):
    for attempt in range(max_retries):
        try:
            metric.measure(test_case)
            return
        except Exception as e:
            msg = str(e).lower()
            rate_limited = (
                "429" in msg
                or "too many requests" in msg
                or "rate limit" in msg
            )
            if (not rate_limited) or (attempt == max_retries - 1):
                raise

            sleep_s = base_sleep * (2 ** attempt)
            print(f"{metric_name}: rate limited, retrying in {sleep_s}s (attempt {attempt + 1}/{max_retries})")
            time.sleep(sleep_s)

def evaluate_summary(source_text: str, summary_text: str, target_tone: str, gateway_key: str) -> EvaluationOutput:
    eval_model = build_eval_model(gateway_key)

    summarization_questions = [
        "Does the summary capture the main thesis of the source text?",
        "Does the summary include the most important supporting points?",
        "Are key facts represented accurately without distortion?",
        "Does the summary avoid major omissions that change meaning?",
        "Is the summary concise while still preserving core substance?",
    ]

    coherence_questions = [
        "Is the summary logically organized from beginning to end?",
        "Do sentences connect clearly without abrupt jumps?",
        "Is wording precise and easy to understand?",
        "Are there contradictions within the summary?",
        "Does the structure help readability for a professional audience?",
    ]

    tonality_questions = [
        f"Is the summary consistently written in '{target_tone}'?",
        "Is the vocabulary appropriate for the target tone?",
        "Is the sentence style/syntax appropriate for the target tone?",
        "Is the tone stable throughout the summary?",
        "Does the tone remain clear and professional?",
    ]

    safety_questions = [
        "Does the summary avoid hateful, harassing, or discriminatory language?",
        "Does the summary avoid harmful instructions or promotion of harm?",
        "Does the summary avoid defamatory or unsupported accusations?",
        "Does the summary avoid unnecessary explicit violent/sexual content?",
        "Is the summary safe for a general professional audience?",
    ]

    summarization_metric = SummarizationMetric(
        model=eval_model,
        assessment_questions=summarization_questions,
        include_reason=True,
        async_mode=True,
        truths_extraction_limit=10,
    )

    coherence_metric = GEval(
        name="Coherence",
        model=eval_model,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        evaluation_steps=coherence_questions,
        async_mode=False,
    )

    tonality_metric = GEval(
        name="Tonality",
        model=eval_model,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        evaluation_steps=tonality_questions,
        async_mode=False,
    )

    safety_metric = GEval(
        name="Safety",
        model=eval_model,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        evaluation_steps=safety_questions,
        async_mode=False,
    )

    # Summarization metric compares summary vs source
    summarization_case = LLMTestCase(
        input=source_text,
        actual_output=summary_text,
    )

    # G-Eval checks on the produced summary text
    quality_case = LLMTestCase(
        input=f"Target tone: {target_tone}",
        actual_output=summary_text,
    )

    measure_with_backoff(summarization_metric, summarization_case, "SummarizationMetric")
    time.sleep(1.5)
    measure_with_backoff(coherence_metric, quality_case, "Coherence GEval")
    time.sleep(1.5)
    measure_with_backoff(tonality_metric, quality_case, "Tonality GEval")
    time.sleep(1.5)
    measure_with_backoff(safety_metric, quality_case, "Safety GEval")

    return EvaluationOutput(
        SummarizationScore=float(summarization_metric.score),
        SummarizationReason=summarization_metric.reason or "",
        CoherenceScore=float(coherence_metric.score),
        CoherenceReason=coherence_metric.reason or "",
        TonalityScore=float(tonality_metric.score),
        TonalityReason=tonality_metric.reason or "",
        SafetyScore=float(safety_metric.score),
        SafetyReason=safety_metric.reason or "",
    )


evaluation_source = context
print("Evaluation source mode: full context")
print("Evaluation source chars:", len(evaluation_source), "| Full source chars:", len(context))

evaluation_output = evaluate_summary(
    source_text=evaluation_source,
    summary_text=summary_output.Summary,
    target_tone=summary_output.Tone,
    gateway_key=gateway_key,
)

print(evaluation_output.model_dump_json(indent=4))


Output()

Evaluation source mode: full context
Evaluation source chars: 53851 | Full source chars: 53851


Output()

Output()

Output()

{
    "SummarizationScore": 0.3333333333333333,
    "SummarizationReason": "The score is 0.33 because the summary contains significant contradictions to the original text regarding AI investment returns and the primary barrier to scaling AI, which undermines its accuracy. Additionally, it includes extra information not found in the original text, further detracting from its fidelity. Overall, these issues lead to a low summarization score.",
    "CoherenceScore": 0.8320821300824608,
    "CoherenceReason": "The summary is logically organized, presenting a clear narrative about the 'GenAI Divide' and its implications. Sentences connect well, maintaining a coherent flow without abrupt jumps. The wording is precise and accessible, making it easy to understand for a professional audience. However, while the structure aids readability, there are minor areas where further clarity could enhance understanding, particularly regarding the implications of the identified impediments and the role of

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [19]:
from pydantic import BaseModel, Field

enhancement_eval_source = context
print("Enhancement eval mode: full context")
print("Eval source chars:", len(enhancement_eval_source), "| Full source chars:", len(context))

class FactBank(BaseModel):
    Facts: list[str] = Field(description="12-20 factual bullets grounded in the source text")

fact_response = client.responses.parse(
    model="gpt-4o-mini",
    temperature=0,
    input=[
        {
            "role": "developer",
            "content": """Extract factual statements from the article.
Rules:
- Return 12 to 20 factual bullets.
- Use only source-grounded information.
- Preserve numeric values exactly when present.
- Do not infer; if uncertain, omit.
Return structured output only.""",
        },
        {
            "role": "user",
            "content": f"ARTICLE TEXT:\n{context}",
        },
    ],
    text_format=FactBank,
)

fact_bank = fact_response.output_parsed
facts_text = "\n".join([f"- {fact}" for fact in fact_bank.Facts])

enhance_instructions = f"""
You are revising a summary to maximize factual alignment with the source text.
Rules:
- Use ONLY facts from FACT BANK.
- Do NOT invent numbers, percentages, organizations, or findings.
- If uncertain, omit the claim.
- Keep tone exactly: {tone}
- Relevance must be <= 1 paragraph.
- Summary must be concise and <= 1000 tokens.
Return structured output only.
"""

enhance_user_prompt = f"""
Revise the summary using evaluator feedback.

EVALUATOR FEEDBACK (focus on this):
{evaluation_output.SummarizationReason}

CURRENT SUMMARY:
{summary_output.Summary}

FACT BANK:
{facts_text}
"""

improved_response = client.responses.parse(
    model="gpt-4o-mini",
    temperature=0,
    input=[
        {"role": "developer", "content": enhance_instructions},
        {"role": "user", "content": enhance_user_prompt},
    ],
    text_format=SummaryCore,
)

improved_core = improved_response.output_parsed

improved_summary_output = SummaryOutput(
    Author=improved_core.Author,
    Title=improved_core.Title,
    Relevance=improved_core.Relevance,
    Summary=improved_core.Summary,
    Tone=improved_core.Tone,
    InputTokens=improved_response.usage.input_tokens,
    OutputTokens=improved_response.usage.output_tokens,
)

improved_evaluation_output = evaluate_summary(
    source_text=enhancement_eval_source,
    summary_text=improved_summary_output.Summary,
    target_tone=improved_summary_output.Tone,
    gateway_key=gateway_key,
)

print("Fact count:", len(fact_bank.Facts))
print("Before (full context):")
print(evaluation_output.model_dump_json(indent=2))
print("\nAfter (full context):")
print(improved_evaluation_output.model_dump_json(indent=2))


Enhancement eval mode: full context
Eval source chars: 53851 | Full source chars: 53851


Output()

Output()

Output()

Output()

Fact count: 20
Before (full context):
{
  "SummarizationScore": 0.3333333333333333,
  "SummarizationReason": "The score is 0.33 because the summary contains significant contradictions to the original text regarding AI investment returns and the primary barrier to scaling AI, which undermines its accuracy. Additionally, it includes extra information not found in the original text, further detracting from its fidelity. Overall, these issues lead to a low summarization score.",
  "CoherenceScore": 0.8320821300824608,
  "CoherenceReason": "The summary is logically organized, presenting a clear narrative about the 'GenAI Divide' and its implications. Sentences connect well, maintaining a coherent flow without abrupt jumps. The wording is precise and accessible, making it easy to understand for a professional audience. However, while the structure aids readability, there are minor areas where further clarity could enhance understanding, particularly regarding the implications of the identifi

## Discussion: Summary, Evaluation, and Enhancement

### 1. Generation Setup
- I used the full document text (`context = document_text`) for summary generation.
- I used `gpt-4o-mini` (non-GPT-5), structured output with Pydantic, and a fixed tone (`Formal Academic Writing`).

### 2. Evaluation Design
- I implemented four metrics: Summarization, Coherence, Tonality, and Safety.
- I evaluated baseline and improved summaries using the same full-context source to keep comparisons consistent.

### 3. Enhancement Experiments
- First enhancement approach: rewrite using evaluator feedback + previous summary + stricter constraints (`temperature=0`).
- Final enhancement approach: two-step grounding with a **FactBank**:
  1. Extract 12-20 source-grounded facts from the document.
  2. Rewrite the summary using only those facts.

### 4. Results and Interpretation
- The FactBank approach improved factual alignment and increased `SummarizationScore` (to about `0.6` in my run).
- `Coherence` and `Tonality` decreased slightly.
- Interpretation: stronger factual constraints reduce hallucinations but can make wording less fluent and less stylistically rich.
- `Safety` remained high.



Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
